# 🛒 Sentiment Analysis — Amazon Product Reviews

**Use Case :** Analyse de sentiments sur les avis clients Amazon  
**Dataset :** `amazon_polarity` (HuggingFace)  
**Modèle :** GPT-4o via OpenAI API  
**Stratégies :** Zero-Shot · Few-Shot · Chain-of-Thought

---

L'objectif est de classifier les avis Amazon en **positif** ou **négatif** en utilisant différentes techniques de Prompt Engineering, puis d'évaluer leur performance via le **score micro-F1**.

## Step 1 : Définir les objectifs & métriques

Dans l'analyse de sentiment, nous attribuons au texte fourni en entrée l'une des deux étiquettes suivantes : **positif** ou **négatif**.

Nous utilisons le dataset `amazon_polarity` qui contient des avis clients sur des produits Amazon. Ce dataset diffère du dataset IMDB (films) par son langage plus **direct et concret** : les avis Amazon contiennent souvent des mots-clés forts (`excellent`, `broken`, `love it`, `waste of money`) qui facilitent la classification.

**Métriques utilisées :**
- **Taux d'exactitude (accuracy)** : nombre de prédictions correctes
- **Score micro-F1** : agrège les TP, FN, FP sur l'ensemble des classes — préféré car plus robuste face aux déséquilibres de classes

## Step 2 : Charger et Analyser le Dataset

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
print("✅ Environnement chargé")

In [ ]:
# Chargement du dataset Amazon Polarity
corpus = load_dataset("amazon_polarity")
corpus

Le dataset `amazon_polarity` est composé de :
- **3 600 000** avis pour le train
- **400 000** avis pour le test
- **Colonnes :** `label` (0=négatif, 1=positif), `title`, `content`

Nous utilisons le champ `content` comme texte d'entrée principal.

In [ ]:
# Convertir en DataFrame et renommer 'content' en 'text'
train_df = corpus['train'].to_pandas()
train_df = train_df.rename(columns={'content': 'text'})
train_df.info()

In [ ]:
# Distribution des labels
train_df.label.value_counts()

In [ ]:
# Ajouter la colonne sentiment textuelle
train_df['sentiment'] = np.where(train_df.label == 1, 'positive', 'negative')
train_df.sample(6)

In [ ]:
# Vérifier l'équilibre des classes
train_df.sentiment.value_counts()

In [ ]:
# Visualiser la distribution
train_df.sentiment.value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'])
plt.title("Distribution des sentiments — Amazon Polarity")
plt.xlabel("Sentiment")
plt.ylabel("Nombre d'avis")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('cap1.png', dpi=150)
plt.show()

## Step 3 : Examples & Gold Examples

Nous divisons les données en deux segments :
- **examples_df** : réservoir pour les exemples few-shot (dynamic sampling)
- **gold_examples_df** : ensemble fixe pour évaluer les prompts

In [ ]:
# Échantillonner pour des temps d'exécution raisonnables
# Amazon Polarity est très grand (3.6M) — on prend 30 000 avis
sampled_df = train_df.sample(n=30000, random_state=42)

examples_df, gold_examples_df = train_test_split(
    sampled_df, test_size=0.2, random_state=42
)
print(f"Examples pool   : {examples_df.shape}")
print(f"Gold examples   : {gold_examples_df.shape}")

In [ ]:
# Sélectionner 20 gold examples équilibrés
columns = ['text', 'sentiment']
gold_examples = (
    gold_examples_df.loc[:, columns]
    .sample(20, random_state=42)
    .to_json(orient='records')
)

# Afficher le premier gold example
print(json.dumps(json.loads(gold_examples)[0], indent=2, ensure_ascii=False))

In [ ]:
# Visualiser la distribution des gold examples
gold_df = pd.DataFrame(json.loads(gold_examples))
gold_df.sentiment.value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'])
plt.title("Distribution des Gold Examples")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('cap2.png', dpi=150)
plt.show()

## Step 4 : Élaborer les Prompts

### Template du message utilisateur

In [ ]:
user_message_template = """```{amazon_review}```"""

### Prompt 1 : Zero-Shot

In [ ]:
zero_shot_system_message = """
Classify the sentiment of Amazon product reviews presented in the input as 'positive' or 'negative'.
Amazon reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'.
Do not explain your answer.
"""

zero_shot_prompt = [{'role': 'system', 'content': zero_shot_system_message}]

### Prompt 2 : Few-Shot

Nous créons des exemples équilibrés (4 positifs + 4 négatifs) organisés aléatoirement pour éviter :
- Le **biais de l'étiquette majoritaire**
- Le **biais de récence** (exemples vers la fin du prompt)

In [ ]:
few_shot_system_message = """
Classify the sentiment of Amazon product reviews presented in the input as 'positive' or 'negative'.
Amazon reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'.
Do not explain your answer.
"""

def create_examples(dataset, n=4):
    """Crée des exemples few-shot équilibrés et mélangés aléatoirement."""
    positive_reviews = (dataset.sentiment == 'positive')
    negative_reviews = (dataset.sentiment == 'negative')
    cols = ['text', 'sentiment']
    pos_examples = dataset.loc[positive_reviews, cols].sample(n)
    neg_examples = dataset.loc[negative_reviews, cols].sample(n)
    examples = pd.concat([pos_examples, neg_examples])
    return examples.sample(2 * n, replace=False).to_json(orient='records')


def create_prompt(system_message, examples, user_message_template):
    """Assemble un prompt few-shot complet."""
    prompt = [{'role': 'system', 'content': system_message}]
    for example in json.loads(examples):
        prompt.append({
            'role': 'user',
            'content': user_message_template.format(amazon_review=example['text'])
        })
        prompt.append({
            'role': 'assistant',
            'content': example['sentiment']
        })
    return prompt


examples = create_examples(examples_df, 2)
few_shot_prompt = create_prompt(few_shot_system_message, examples, user_message_template)
few_shot_prompt

### Prompt 3 : Chain-of-Thought (CoT)

In [ ]:
cot_system_message = """
Classify the sentiment of Amazon product reviews presented in the input as 'positive' or 'negative'.
Amazon reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'.
Do not explain your answer.

Instructions:
1. Carefully read the review and identify key sentiment signals.
2. Consider strong indicator words: (excellent, love, perfect, broken, terrible, waste, disappointed...)
3. Consider the overall tone and context of the review.
4. Estimate the probability of the review being positive.

To reiterate, your answer should strictly only contain the label: positive or negative.
"""

cot_few_shot_prompt = create_prompt(cot_system_message, examples, user_message_template)
cot_few_shot_prompt

## Step 5 : Évaluation des Prompts

In [ ]:
def evaluate_prompt(prompt, gold_examples, user_message_template, model):
    """Évalue un prompt sur les gold examples — retourne le score micro-F1."""
    model_predictions, ground_truths = [], []

    for example in json.loads(gold_examples):
        user_input = [{
            'role': 'user',
            'content': user_message_template.format(amazon_review=example['text'])
        }]
        try:
            response = model.invoke(prompt + user_input)
            response_content = response.content if hasattr(response, 'content') else response
            content_lower = response_content.strip().lower()

            if 'negative' in content_lower:
                prediction = 'negative'
            elif 'positive' in content_lower:
                prediction = 'positive'
            else:
                prediction = 'unknown'

            model_predictions.append(prediction)
            ground_truths.append(example['sentiment'])
        except Exception as e:
            print(f"Erreur : {e}")
            continue

    return f1_score(ground_truths, model_predictions, average='micro')


# Initialiser le modèle
gpt = ChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:
# Zero-Shot
zero_score = evaluate_prompt(zero_shot_prompt, gold_examples, user_message_template, gpt)
print(f"Zero-Shot  micro-F1 : {zero_score:.4f}")

In [ ]:
# Few-Shot
few_score = evaluate_prompt(few_shot_prompt, gold_examples, user_message_template, gpt)
print(f"Few-Shot   micro-F1 : {few_score:.4f}")

In [ ]:
# CoT
cot_score = evaluate_prompt(cot_few_shot_prompt, gold_examples, user_message_template, gpt)
print(f"CoT        micro-F1 : {cot_score:.4f}")

## Step 6 : Analyse de Stabilité — 10 Runs

In [ ]:
num_eval_runs = 10
few_shot_performance, cot_few_shot_performance = [], []

for _ in tqdm(range(num_eval_runs)):
    examples = create_examples(examples_df)

    # Few-Shot
    fs_prompt = create_prompt(few_shot_system_message, examples, user_message_template)
    fs_f1 = evaluate_prompt(fs_prompt, gold_examples, user_message_template, gpt)
    few_shot_performance.append(fs_f1)

    # CoT
    cot_prompt = create_prompt(cot_system_message, examples, user_message_template)
    cot_f1 = evaluate_prompt(cot_prompt, gold_examples, user_message_template, gpt)
    cot_few_shot_performance.append(cot_f1)

In [ ]:
mean_fs  = np.array(few_shot_performance).mean()
std_fs   = np.array(few_shot_performance).std()
mean_cot = np.array(cot_few_shot_performance).mean()
std_cot  = np.array(cot_few_shot_performance).std()

print(f"Few-Shot  — F1 Moyen : {mean_fs:.4f} | Std : {std_fs:.2e}")
print(f"CoT       — F1 Moyen : {mean_cot:.4f} | Std : {std_cot:.2e}")

In [ ]:
# Visualisation de la stabilité
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(few_shot_performance, marker='o', label='Few-Shot', color='steelblue')
ax.plot(cot_few_shot_performance, marker='s', label='CoT Few-Shot', color='darkorange')
ax.axhline(mean_fs,  color='steelblue',  linestyle='--', alpha=0.6, label=f'Moy Few-Shot : {mean_fs:.3f}')
ax.axhline(mean_cot, color='darkorange', linestyle='--', alpha=0.6, label=f'Moy CoT : {mean_cot:.3f}')
ax.set_xlabel("Run")
ax.set_ylabel("Micro F1-Score")
ax.set_title("Stabilité des Prompts — Amazon Polarity (10 runs)")
ax.legend()
ax.set_ylim([0.7, 1.05])
plt.tight_layout()
plt.savefig('imgsentiment1.png', dpi=150)
plt.show()

## Synthèse des Résultats

| Stratégie | F1-Score Moyen | Std | Stabilité |
|-----------|---------------|-----|----------|
| Zero-Shot | — | — | — |
| Few-Shot  | — | — | ✅ |
| CoT       | — | — | ✅ |

**Conclusions — Amazon vs IMDB :**
- Le langage Amazon est plus **direct** → meilleure performance Zero-Shot
- Les mots-clés forts (`broken`, `love it`, `waste`) facilitent la classification
- Le CoT améliore l'interprétabilité sans sacrifier les performances
- La stabilité sur 10 runs confirme la robustesse des stratégies few-shot